In [1]:
# Uninstall original Albumentations if installed
!uv pip uninstall albumentations

Using Python 3.12.12 environment at: /usr


In [2]:
!uv pip install -q xformers torchvision torchmetrics albumentationsx

- DINOv2 models that include "reg" in their name use 4 register tokens.

- For models without register tokens (num_register_tokens = 0):
[ CLS token | Patch token 1 | Patch token 2 | ... | Patch token N ]

- For models with register tokens (num_register_tokens > 0):
[ CLS token | Register token 1 | ... | Register token K | Patch token 1 | Patch token 2 | ... | Patch token N ]
(where K is num_register_tokens)

**ImageNet normalization values:**

- Mean: [0.485, 0.456, 0.406] for RGB channels respectively
- Standard deviation: [0.229, 0.224, 0.225] for RGB channels respectively

In [3]:
import os
import logging
import random
from dataclasses import dataclass, field
from pathlib import Path

from typing import List, Optional, Callable, Dict, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

from torchmetrics import Accuracy
from sklearn.metrics import confusion_matrix, classification_report

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from tqdm.notebook import tqdm  # Better progress bars for notebooks
print("albumentations' version: ", A.__version__)

albumentations' version:  2.0.13


In [4]:
# mount drive
import os
from google.colab import drive
drive.mount('/content/drive')

# Change the current working directory
new_dir = "/content/drive/MyDrive/"
os.chdir(new_dir)

# Get the current working directory
current_dir = os.getcwd()
print(f"Current working directory after changing: {current_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working directory after changing: /content/drive/MyDrive


In [5]:
# ----------------------------
# Hardware Check
# ----------------------------

def check_hardware() -> torch.device:
    if not torch.cuda.is_available():
        logger.warning("No GPU detected. Using CPU.")
        return torch.device("cpu")
    device_index = torch.cuda.current_device()
    device = torch.device(f"cuda:{device_index}")
    name = torch.cuda.get_device_name()
    mem = torch.cuda.get_device_properties().total_memory / 1024**3
    print(f"Training on GPU: {name} with {mem:.1f} GB")
    return device

import multiprocessing
# sklearn and multiprocessing
cores = multiprocessing.cpu_count()
print(f"Number of CPU cores: {cores}")
check_hardware()

Number of CPU cores: 8
Training on GPU: Tesla T4 with 14.7 GB


device(type='cuda', index=0)

In [6]:
#@title DinoV2 reg

# ----------------------------
# Logger Setup
# ----------------------------
LOGGER_NAME = "dinov2"
logger = logging.getLogger(LOGGER_NAME)

def setup_logging(log_level: str = "INFO") -> None:
    logger.setLevel(getattr(logging, log_level.upper(), logging.INFO))
    if logger.hasHandlers():
        for h in list(logger.handlers):
            logger.removeHandler(h)
    ch = logging.StreamHandler()
    ch.setLevel(logger.level)
    ch.setFormatter(logging.Formatter("%(asctime)s %(levelname)s: %(message)s"))
    logger.addHandler(ch)
    logger.propagate = False

setup_logging()

# ----------------------------
# Configuration
# ----------------------------
@dataclass
class Config:
    model_name: str = "dinov2_vitb14_reg"
    epochs: int = 20
    batch_size: int = 32
    num_workers: int = 8
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    dropout_rate: float = 0.5
    scheduler_factor: float = 0.1
    scheduler_patience: int = 5
    early_stopping_patience: int = 5
    seed: int = 123
    data_dir: Path = Path("./batdrive/OC/P6/P6_data/Images/")
    data_csv: Path = Path("./batdrive/OC/P6/P6_data/df_cleaned.csv")
    output_dir: Path = Path("./batdrive/models/outputs/dino/")
    save_all_misclassified: bool = True
    required_columns: list = field(default_factory=lambda: ["image", "level_1"])

    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------
# Data Loading & Validation
# ----------------------------
def validate_csv_columns(csv_path: Path, required_columns: list) -> None:
    df_cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
    missing = set(required_columns) - set(df_cols)
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}")

def load_dataframe(csv_path: Path, required_columns: list) -> pd.DataFrame:
    validate_csv_columns(csv_path, required_columns)
    return pd.read_csv(csv_path, usecols=required_columns)

class ImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: A.Compose, data_dir: Path):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.data_dir = data_dir

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        img_filename = self.df.loc[idx, "image"]
        img_path = self.data_dir / img_filename
        label = int(self.df.loc[idx, "label_encoded"])
        image = cv2.imread(str(img_path))
        if image is None:
            raise FileNotFoundError(f"Image at path {img_path} could not be loaded.")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transform(image=image)["image"]
        return image, label, img_filename

def get_simple_product_transforms(is_training: bool, image_size: int = 224) -> A.Compose:
    """
    augmentation pipeline for product classification
    """
    if is_training:
        # Key augmentations for product images
        return A.Compose([
            # 1. Resize and crop to a consistent size.
            A.SmallestMaxSize(max_size=256, interpolation=cv2.INTER_AREA),
            A.RandomCrop(height=image_size, width=image_size),

            # 2. HorizontalFlip
            A.HorizontalFlip(p=0.5),

            # 3. Color augmentation is very important for handling different product colors
            # and lighting conditions
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.8),

            # 4. Normalization
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    else:
        # validation pipeline: just resize, crop, and normalize
        return A.Compose([
            A.SmallestMaxSize(max_size=image_size, interpolation=cv2.INTER_AREA),
            A.CenterCrop(height=image_size, width=image_size),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])

# ----------------------------
# Reproducibility & Hardware
# ----------------------------
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        cudnn.deterministic = False
        cudnn.benchmark = True

def check_hardware() -> torch.device:
    if not torch.cuda.is_available():
        logger.warning("No GPU detected. Using CPU.")
        return torch.device("cpu")
    device = torch.device("cuda", torch.cuda.current_device())
    name = torch.cuda.get_device_name(device)
    mem = torch.cuda.get_device_properties(device).total_memory / 1024**3
    print(f"Training on GPU: {name} ({mem:.1f} GB)")
    return device

# ----------------------------
# Model Definition
# ----------------------------
class DINOv2Classifier(nn.Module):
    def __init__(self, num_classes: int, dropout_rate: float):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', Config.model_name)
        for p in self.backbone.parameters():
            p.requires_grad = False
        embed_dim = self.backbone.embed_dim
        self.head = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(embed_dim, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(x)
        return self.head(feats)

# ----------------------------
# Early Stopping & Metrics
# ----------------------------
class EarlyStopping:
    def __init__(self, patience: int, delta: float, save_path: Path):
        self.patience = patience
        self.delta = delta
        self.save_path = save_path
        self.best_loss = float("inf")
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss: float, model: nn.Module) -> None:
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            # save only the head's weights
            torch.save(model.head.state_dict(), str(self.save_path))
        else:
            self.counter += 1
            logger.info(f"EarlyStopping counter {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

# ----------------------------
# Training & Validation Loops
# ----------------------------
def train_epoch(model, loader, criterion, optimizer, device, acc_metric):
    model.train()
    losses, accs = 0.0, 0.0
    for inputs, labels, _ in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        losses += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        acc_metric.update(preds, labels)
    total = len(loader.dataset)
    return losses/total, acc_metric.compute().item()

def validate(model, loader, criterion, device, acc_metric):
    model.eval()
    loss_total = 0.0
    mis = {"filenames": [], "true": [], "pred": []}
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels, filenames in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss_total += loss.item() * inputs.size(0)
            preds = outputs.argmax(dim=1)
            acc_metric.update(preds, labels)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

            diff_idxs = (preds != labels).nonzero(as_tuple=True)[0]
            for i in diff_idxs.cpu().tolist():
                if len(mis["filenames"]) >= 1000:
                    break
                mis["filenames"].append(filenames[i])
                mis["true"].append(labels[i].item())
                mis["pred"].append(preds[i].item())

    total = len(loader.dataset)
    val_loss = loss_total / total
    val_acc = acc_metric.compute().item()
    acc_metric.reset()

    logger.info(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    return {
        "val_loss": val_loss,
        "val_acc": val_acc,
        "misclassified": mis,
        "all_preds": all_preds,
        "all_labels": all_labels,
    }

# ----------------------------
# Plotting Helpers
# ----------------------------
def plot_learning_curves(history: dict, out: Path) -> None:
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure()
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.savefig(out / "loss_curve.png", dpi=300)
    plt.close()

    plt.figure()
    plt.plot(epochs, history["train_acc"], label="Train Acc")
    plt.plot(epochs, history["val_acc"], label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.savefig(out / "acc_curve.png", dpi=300)
    plt.close()


def plot_confusion_matrix(all_labels, all_preds, class_names, output_dir: Path) -> None:
    cm = confusion_matrix(all_labels, all_preds)
    n = len(class_names)
    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    fig.colorbar(im, ax=ax)
    ax.set_xticks(range(n)); ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticks(range(n)); ax.set_yticklabels(class_names)
    ax.set_title('Confusion Matrix')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    thresh = cm.max() / 2.0
    for i in range(n):
        for j in range(n):
            color = 'white' if cm[i, j] > thresh else 'black'
            ax.text(j, i, f"{cm[i, j]:d}", ha='center', va='center', color=color)
    fig.tight_layout()
    fig.savefig(output_dir / 'confusion_matrix.png', dpi=300)
    plt.close(fig)


def plot_misclassified(mis: dict, class_names: list, data_dir: Path, output_dir: Path, save_all: bool) -> None:
    n_total = len(mis['filenames'])
    n = n_total if save_all else min(16, n_total)
    if n == 0:
        logger.info("No misclassified samples to plot.")
        return
    cols = min(4, n)
    rows = (n + cols - 1) // cols
    fig = plt.figure(figsize=(cols * 4, rows * 4))
    for idx in range(n):
        ax = fig.add_subplot(rows, cols, idx + 1)
        img_path = data_dir / mis['filenames'][idx]
        img = cv2.imread(str(img_path))
        if img is None:
            ax.text(0.5, 0.5, "Image not found", ha="center", va="center")
            ax.axis("off")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.axis("off")
        true_lbl = class_names[mis['true'][idx]]
        pred_lbl = class_names[mis['pred'][idx]]
        title_lines = [f"True: {true_lbl}\nPred: {pred_lbl}"]
        if save_all or n == 1:
            title_lines.append(f"File: {mis['filenames'][idx]}")
        ax.set_title("\n".join(title_lines), fontsize=9, pad=8, loc="center")
    plt.tight_layout()
    plt.savefig(output_dir / "misclassified_samples.png", dpi=300)
    plt.close()

# ----------------------------
# Model Training & Evaluation
# ----------------------------
def train_model(config: Config) -> nn.Module:
    set_seed(config.seed)
    df = load_dataframe(config.data_csv, config.required_columns)
    le = LabelEncoder()
    df["label_encoded"] = le.fit_transform(df["level_1"])
    class_names = le.classes_.tolist()
    num_classes = len(class_names)

    train_df, val_df = train_test_split(df, test_size=0.2,
                                        stratify=df["label_encoded"],
                                        random_state=config.seed)
    train_loader = DataLoader(
        ImageDataset(train_df, get_simple_product_transforms(True), config.data_dir),
        batch_size=config.batch_size, shuffle=True,
        num_workers=config.num_workers, pin_memory=True)
    val_loader = DataLoader(
        ImageDataset(val_df, get_simple_product_transforms(False), config.data_dir),
        batch_size=config.batch_size, shuffle=False,
        num_workers=config.num_workers, pin_memory=True)

    device = check_hardware()
    model = DINOv2Classifier(num_classes, config.dropout_rate).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.head.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                     factor=config.scheduler_factor,
                                                     patience=config.scheduler_patience)
    early_stopper = EarlyStopping(config.early_stopping_patience, delta=1e-4,
                                  save_path=config.output_dir / "best_head.pth")
    train_acc = Accuracy(task="multiclass", num_classes=num_classes).to(device)
    val_acc   = Accuracy(task="multiclass", num_classes=num_classes).to(device)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    logger.info("=== Training head only ===")
    for epoch in range(1, config.epochs + 1):
        tr_loss, tr_a = train_epoch(model, train_loader, criterion, optimizer, device, train_acc)
        train_acc.reset()
        val_res = validate(model, val_loader, criterion, device, val_acc)
        scheduler.step(val_res["val_loss"])

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_a)
        history["val_loss"].append(val_res["val_loss"])
        history["val_acc"].append(val_res["val_acc"])

        logger.info(f"Epoch {epoch}/{config.epochs} — Train Loss: {tr_loss:.4f},"
                    f" Acc: {tr_a:.4f} | Val Loss: {val_res['val_loss']:.4f},"
                    f" Acc: {val_res['val_acc']:.4f}")
        early_stopper(val_res["val_loss"], model)
        if early_stopper.early_stop:
            logger.info("Early stopping triggered.")
            break

    # Load best head only
    logger.info("Loading best head weights from disk")
    head_state = torch.load(config.output_dir / "best_head.pth", map_location=device, weights_only=True)
    model.head.load_state_dict(head_state)

    # --- Final evaluation & plotting ---
    logger.info("=== Final Evaluation ===")
    final_res = validate(model, val_loader, criterion, device, Accuracy(task="multiclass", num_classes=num_classes).to(device))

    # Save misclassified samples
    mis = final_res["misclassified"]
    df_mis = pd.DataFrame({
        "True class": [class_names[i] for i in mis["true"]],
        "Predicted class": [class_names[i] for i in mis["pred"]]
    }, index=mis["filenames"])
    df_mis.index.name = "Image File"
    df_mis.to_csv(config.output_dir / "misclassified_samples.csv")
    logger.info(f"Saved misclassified samples to {config.output_dir / 'misclassified_samples.csv'}")

    # Save classification report
    report = classification_report(final_res["all_labels"], final_res["all_preds"], target_names=class_names)
    with open(config.output_dir / "classification_report.txt", "w") as f:
        f.write(report)
    logger.info(f"Saved classification report to {config.output_dir / 'classification_report.txt'}")

    # Plot & save figures
    plot_learning_curves(history, config.output_dir)
    plot_confusion_matrix(final_res["all_labels"], final_res["all_preds"], class_names, config.output_dir)
    plot_misclassified(mis, class_names, config.data_dir, config.output_dir, config.save_all_misclassified)

    return model

if __name__ == "__main__":
    cfg = Config(save_all_misclassified=True, epochs=25)
    model = train_model(cfg)

Training on GPU: Tesla T4 (14.7 GB)


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")
2025-11-21 17:59:03,354 INFO: using MLP layer as FFN
2025-11-21 17:59:05,101 INFO: === Training head only ===
2025-11-21 17:59:22,877 INFO: Val Loss: 1.7404, Val Acc: 0.3667
2025-11-21 17:59:22,879 INFO: Epoch 1/25 — Train Loss: 2.0681, Acc: 0.1952 | Val Loss: 1.7404, Acc: 0.3667
2025-11-21 17:59:39,885 INFO: Val Loss: 1.5026, Val Acc: 0.5381
2025-11-21 17:59:39,886 INFO: Epoch 2/25 — Train L

In [7]:
#@title Résultats

# 1. Définition du chemin
output_dir = Path("./batdrive/models/outputs/dino/")

print(f"Lecture des résultats depuis : {output_dir.resolve()}\n")

# ---------------------------------------------------------
# 2. Affichage du DataFrame des erreurs
# ---------------------------------------------------------
csv_path = output_dir / "misclassified_samples.csv"
if csv_path.exists():
    display(Markdown("### Misclassified samples"))
    df_mis = pd.read_csv(csv_path, index_col=0)
    display(df_mis)
else:
    print(f"Fichier non trouvé : {csv_path}")

# ---------------------------------------------------------
# 3. Affichage du rapport de classification
# ---------------------------------------------------------
report_path = output_dir / "classification_report.txt"
if report_path.exists():
    display(Markdown("### Classification Report"))
    with open(report_path, "r") as f:
        print(f.read())
else:
    print(f"Fichier non trouvé : {report_path}")

# ---------------------------------------------------------
# 4. Affichage des graphiques
# ---------------------------------------------------------
image_files = [
    ("loss_curve.png", "Learning curves (Loss)"),
    ("acc_curve.png", "Learning curves (Accuracy)"),
    ("confusion_matrix.png", "Confusion matrix"),
    ("misclassified_samples.png", "Misclassified images")
]

for filename, title in image_files:
    img_path = output_dir / filename
    if img_path.exists():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(img_path)))
        print("\n")
    else:
        print(f"Image non trouvée : {filename}")

Output hidden; open in https://colab.research.google.com to view.